# 🤖 Addestra il TUO modello AI (tutto nel cloud ☁️)

Questo notebook addestra un piccolo modello di linguaggio (GPT) **da zero**, usando solo il cloud di Google Colab.
**Niente viene salvato o eseguito sul tuo computer.**

### Come usarlo
1. Apri questo notebook in Google Colab (pulsante "Open in Colab" nel README su GitHub).
2. Menu **Runtime → Change runtime type → T4 GPU** (gratis).
3. Menu **Runtime → Run all**.
4. Alla fine il modello viene salvato e puoi usarlo per generare testo.

> 💡 Il training richiede ~10-20 minuti su GPU gratuita. Non chiudere la scheda durante l'addestramento.


## 1. Installazione dipendenze (nel cloud)

In [ ]:
!pip install -q datasets transformers accelerate huggingface_hub

## 2. Login Hugging Face (opzionale, per salvare il modello online)

Se vuoi salvare il modello sul tuo account Hugging Face (gratis), esegui la cella e incolla il tuo token.
Altrimenti salta: il modello verrà salvato in Colab e potrai scaricarlo.


In [ ]:
from huggingface_hub import notebook_login
# Decommenta la riga sotto se vuoi caricare il modello su Hugging Face Hub
# notebook_login()

## 3. Configurazione (modifica qui se vuoi)

In [ ]:
# Parametri del modello e del training
MODEL_NAME = "mio-modello-ai"          # nome del tuo modello
DATASET = "roneneldan/TinyStories"     # dataset di storie semplici (inglese)
N_TRAIN = 8000                         # numero di storie per il training
N_VAL = 1000                           # numero di storie per la validazione
BLOCK_SIZE = 128                       # lunghezza delle sequenze
BATCH_SIZE = 16                        # riduci a 8 se esce "out of memory"
EPOCHS = 2                             # numero di epoche
LEARNING_RATE = 5e-4
N_LAYER = 4                            # layer del transformer
N_HEAD = 4                             # attention heads
N_EMBD = 256                           # dimensione embedding


## 4. Import librerie

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    GPT2Config, GPT2LMHeadModel, GPT2TokenizerFast,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling,
)
print("PyTorch:", torch.__version__)
print("GPU disponibile:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 5. Caricamento dataset (dal cloud)

In [ ]:
ds = load_dataset(DATASET)
train_raw = ds["train"].select(range(N_TRAIN))
val_raw = ds["validation"].select(range(N_VAL)) if "validation" in ds else ds["train"].select(range(N_TRAIN, N_TRAIN + N_VAL))
print("Train:", len(train_raw), "Val:", len(val_raw))
print(train_raw[0]["text"][:300])

## 6. Tokenizer (riusiamo quello di GPT-2, nessun training necessario)

In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
print("Vocab size:", tokenizer.vocab_size)

## 7. Tokenizzazione

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"])

tokenized_train = train_raw.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_val = val_raw.map(tokenize_function, batched=True, remove_columns=["text"])
print(tokenized_train)

## 8. Raggruppamento in blocchi di testo

In [ ]:
def group_texts(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated[list(examples.keys())[0]])
    total_length = (total_length // BLOCK_SIZE) * BLOCK_SIZE
    result = {k: [t[i:i+BLOCK_SIZE] for i in range(0, total_length, BLOCK_SIZE)]
              for k, t in concatenated.items()}
    result["labels"] = result["input_ids"].copy()
    return result

lm_train = tokenized_train.map(group_texts, batched=True, batch_size=1000)
lm_val = tokenized_val.map(group_texts, batched=True, batch_size=1000)
print("Train blocks:", len(lm_train), "Val blocks:", len(lm_val))

## 9. Creazione del modello (da zero, pesi casuali)

In [ ]:
config = GPT2Config(
    vocab_size=tokenizer.vocab_size,
    n_positions=BLOCK_SIZE,
    n_embd=N_EMBD,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    n_inner=N_EMBD * 4,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)
model = GPT2LMHeadModel(config)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parametri del modello: {n_params/1e6:.2f} M")

## 10. Configurazione training

In [ ]:
training_args = TrainingArguments(
    output_dir=f"./{MODEL_NAME}",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    push_to_hub=False,
)

## 11. Addestramento 🚀

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_train,
    eval_dataset=lm_val,
    data_collator=data_collator,
)

trainer.train()
print("✅ Training completato!")

## 12. Valutazione

In [ ]:
import math
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

## 13. Prova il modello (genera testo)

In [ ]:
prompt = "Once upon a time"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.8, top_p=0.9)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 14. Salvataggio del modello

Il modello è addestrato. Ora puoi:
- **Scaricarlo** dal pannello file di Colab (cartella `mio-modello-ai/`).
- **Caricarlo su Hugging Face Hub** (decommenta `notebook_login()` nella cella 2 e poi esegui la cella sotto).


In [ ]:
# Salva localmente in Colab (poi scarica o carica su HF)
save_path = f"./{MODEL_NAME}-final"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print("Modello salvato in:", save_path)

# Per caricare su Hugging Face Hub (dopo notebook_login()):
# trainer.push_to_hub(repo_name=MODEL_NAME)

## 15. Come usare il modello addestrato dopo

```python
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
model = GPT2LMHeadModel.from_pretrained("percorso/o/hf-id")
tok = GPT2TokenizerFast.from_pretrained("percorso/o/hf-id")
```

Congratulazioni! Hai creato il tuo modello AI, tutto nel cloud. ☁️🎉
